# C2 — Pipeline Supervisionado

Pré-processamento + partição persistida. Consome o artefato do Notebook 03;
não cria partição e mantém o teste fechado.

**Pré-requisito:** `pip install -e .`

In [ ]:
from pathlib import Path
import json
import os
import sys


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import joblib
import pandas as pd
from scipy import sparse
from sklearn.base import clone

from src.utils import config as C
from src.utils import env
from src.utils.performance import Monitor
from src.utils.reproducibility import Run
from src.data.supervised_artifact import load_supervised_artifact, split_partitions
from src.evaluation.leakage import check_leakage
from src.models.pipeline import build_pipeline, filtrar_rotulos_validos

env.carregar_dotenv()
env.garantir_diretorios()

print(C.resumo())

## Carregar artefato (treino + validação)


In [ ]:
EXPECTED_KIND = "provisional"
os.environ["SRAG_COHORT_KIND"] = EXPECTED_KIND

try:
    df, manifest = load_supervised_artifact(
        ROOT_DIR,
        expected_kind=EXPECTED_KIND,
        partitions=("treino", "validacao"),
    )
except FileNotFoundError as exc:
    nomes = {"dataset_supervisionado.parquet", "manifesto_supervisionado.json"}
    encontrados = [
        str(p)
        for p in ROOT_DIR.rglob("*")
        if p.is_file() and p.name in nomes
    ]

    detalhe = (
        f"Arquivos encontrados: {encontrados}"
        if encontrados
        else f"Nenhum dos arquivos esperados foi encontrado em {ROOT_DIR}."
    )
    encontrados_nomes = {Path(p).name for p in encontrados}
    ausentes = sorted(nomes - encontrados_nomes)

    raise FileNotFoundError(
        "O artefato supervisionado está incompleto. "
        f"Arquivos ausentes: {ausentes}. "
        "Execute o Notebook 03 adaptado para gerar o "
        "dataset_supervisionado.parquet e o manifesto correspondente. "
        f"{detalhe}"
    ) from exc
treino, validacao = split_partitions(df, manifest)

PARTICAO = manifest["partition_column"]
TARGET   = manifest.get("target_column") or C.cfg("target", "coluna")
FEATURES = list(manifest.get("feature_columns") or C.features_finais())
PROIBIDAS = manifest.get("forbidden_columns") or list(C.cfg("features", "proibidas") or [])

N_TREINO = int(manifest["partition_counts"]["treino"])
N_VAL    = int(manifest["partition_counts"]["validacao"])
N_TESTE  = int(manifest["partition_counts"]["teste"])

assert set(df[PARTICAO].unique()) == {"treino", "validacao"}
assert set(FEATURES).issubset(df.columns)

print(f"treino    : {N_TREINO:,}")
print(f"validacao : {N_VAL:,}")
print(f"teste     : {N_TESTE:,} (não carregado)")
print(f"features  : {len(FEATURES)}")
print(f"proibidas : {len(PROIBIDAS)}")

## X / y e checagem de leakage


In [ ]:
X_train, y_train = filtrar_rotulos_validos(
    treino[FEATURES],
    pd.to_numeric(treino[TARGET], errors="coerce").astype("float64"),
)
X_val, y_val = filtrar_rotulos_validos(
    validacao[FEATURES],
    pd.to_numeric(validacao[TARGET], errors="coerce").astype("float64"),
)

check_leakage(X_train, forbidden_columns=PROIBIDAS)
check_leakage(X_val,   forbidden_columns=PROIBIDAS)

print(f"treino    : {len(X_train):,}  positivos={int(y_train.sum()):,}")
print(f"validacao : {len(X_val):,}  positivos={int(y_val.sum()):,}")

## Construir e ajustar o pipeline


In [ ]:
pipeline = build_pipeline(X=X_train, y=y_train)
pipeline_arquitetura = clone(pipeline)

monitor = Monitor(notebook="C_11_pipeline_supervisionado")

with monitor.etapa("fit_treino"):
    pipeline.fit(X_train, y_train)

with monitor.etapa("transform_val_10k"):
    X_val_proc_sample = pipeline.transform(X_val.head(10_000))

assert sparse.issparse(X_val_proc_sample), (
    f"pré-processador deveria ser esparso; veio {type(X_val_proc_sample).__name__}"
)
assert X_val_proc_sample.dtype.name == "float32", (
    f"pré-processador deveria produzir float32; veio {X_val_proc_sample.dtype}"
)

N_FEATURES_PROCESSADAS = int(X_val_proc_sample.shape[1])

print(f"features : {X_train.shape[1]} -> {N_FEATURES_PROCESSADAS}")
print(
    f"amostra  : {X_val_proc_sample.shape}  "
    f"nnz={sparse.csr_matrix(X_val_proc_sample).nnz:,}"
)
print(f"monitor  : {monitor.salvar()}")

monitor.resumo()

## Persistir


In [ ]:
OUT_DIR = C.OUTPUTS_DIR / "C2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pipeline_path     = OUT_DIR / "preprocessor.joblib"
pipeline_fit_path = OUT_DIR / "preprocessor_ajustado.joblib"
split_info_path   = OUT_DIR / "split_info.json"

joblib.dump(pipeline_arquitetura, pipeline_path)
joblib.dump(pipeline, pipeline_fit_path)

split_info = {
    "origem":            "artefato_supervisionado",
    "cohort_kind":       manifest.get("cohort_kind"),
    "partition_column":  PARTICAO,
    "n_total":           int(manifest["rows"]),
    "n_treino_split":    N_TREINO,
    "n_val_split":       N_VAL,
    "n_teste_split":     N_TESTE,
    "n_treino_rotulado": int(len(y_train)),
    "n_val_rotulado":    int(len(y_val)),
}
split_info_path.write_text(
    json.dumps(split_info, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Arquitetura : {pipeline_path}")
print(f"Ajustado    : {pipeline_fit_path}")
print(f"Split info  : {split_info_path}")

## Registar execução


In [ ]:
pre          = pipeline.named_steps["pre"]
features_num = list(pre.transformers_[0][2])
features_cat = list(pre.transformers_[1][2])

with Run(modelo="C2_pipeline", notas="Arquitetura do pipeline supervisionado") as r:
    r.add_dataset(
        n_rows=int(manifest["rows"]),
        train_rows=int(len(y_train)),
        valid_rows=int(len(y_val)),
        test_rows=N_TESTE,
    )
    r.features = {
        "n_features":  len(features_num) + len(features_cat),
        "numericas":   features_num,
        "categoricas": features_cat,
    }
    r.parametros = {
        "target":           TARGET,
        "partition_column": PARTICAO,
        "origem_particao":  "artefato_supervisionado",
        "cohort_kind":      manifest.get("cohort_kind"),
    }
    r.metrica(n_features_processadas=N_FEATURES_PROCESSADAS)

print("Run registrado em:", r.pasta)